[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APA-Technology-Division/apa-technology-webinar-resources/blob/main/NPC%202026%20-%20AI%20Hub%20Session/Tech_Division_NPC26_AI_Hub_Session.ipynb)
# Survey Data Analysis with Generative AI
*NPC 2026 AI Hub Demo*

---
**CONTENTS:**
- Part 0 - Environment setup
- Part I -  Exploration of small datasets using chat-based LLM utilities
- Part II -  Methods for robust analysis of large and/or continually updated datasets using chat-based LLM utilities

---
<br>

**This resource was created by the American Planning Association Technology Division**, April 2026. For more information about the Division visit **[tech.planning.org](https://tech.planning.org)**

>*Note that Generative AI was used to develop and debug this demo. No code or copy was incorporated without  vetting and review. If you would like to adapt the code in this notebook for your own purposes, you are encouraged to do so.  Please be sure to provide attribution in accordance with repo's licensing.*

---
## Part 0: Set up your environment

If you are accessing this notebook directly through Google Colab you will need to load two datasets into your current runtime environment.

Run the code cell below to load in dependencies using pip.  This will ensure that your code will actually run and that you are able to access the files you need in order to follow-along during the demo exercise.

Note: If you're running your code locally with Jupyter, the packages may already be installed.

In [ ]:
import subprocess, sys

packages = [
    'pandas',
    'plotly',
    'vaderSentiment',
    'scikit-learn',
    'wordcloud',
    'pyvis',
    'rapidfuzz',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os
import urllib.request
import pandas as pd
import plotly.graph_objects as go
import textwrap
import plotly.graph_objects as go
import subprocess, sys
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from rapidfuzz import process, fuzz

SMALL_DATA_PATH = 'data/chatbot_public_feedback.csv'
LARGE_DATA_PATH = 'data/survey_results.csv'

GITHUB_BASE_URL = 'https://raw.githubusercontent.com/APA-Technology-Division/apa-technology-webinar-resources/85826b0c7815cac706e9352b3757bf51086e5866/NPC%202026%20-%20AI%20Hub%20Session/data/'

os.makedirs('data', exist_ok=True)

for path in [SMALL_DATA_PATH, LARGE_DATA_PATH]:
    if os.path.exists(path):
        print(f'{path}: found')
    else:
        filename = os.path.basename(path)
        url = GITHUB_BASE_URL + filename
        urllib.request.urlretrieve(url, path)
        print(f'{path}: downloaded from GitHub')

## Part 1: Small Dataset Exploration
In this first part you will be working with a small dataset containing 25 questionnaire responses collected over 7 days from users of a newly-launched chatbot tool.  This is the same data that you will upload to your chat-based LLM tool.

As a reminder, when we originally made this dataset we set out to address a few specific points of concern:
1. **(User satisfaction)** How satisfied are users with the chatbot? (responses scaled 1-5)
2. **(Issue rate)** Did the user encounter any issues with the response (0/1)
3. **(Diagnosis)** If yes, what type of issue did they encounter? (short answer)


###Step 1: Review LLM Output for Small Dataset Summarization
- Prompt ChatGPT, Claude, Gemini or your chat-based LLM of choice with the following text.

- Be sure to attach the `survey_results.csv` file before hitting submit:

**Prompt Text:**
```
Based on this data, tell me about:
1) User satisfaction over time
2) The rate at which issues are appearing
3) The types of issues that are appearing
```

### Step 2: Consider the output.  How does it relate to the flowchart?

![ai_flowchart_2026.jpg](https://raw.githubusercontent.com/APA-Technology-Division/apa-technology-webinar-resources/main/NPC%202026%20-%20AI%20Hub%20Session/ai_flowchart_2026.jpg)

###Step 3: Compare Output of Analysis
In this step, you'll plot the data from the small dataset and compare the outputs from the chat-based LLM with analysis using more conventional techniques.  The technique applied is described in brief in the markdown cell that precedes the code.

Begin by running the code cell below to read in the small dataset's csv data.

Once you have run the code.  Take note of the names and contents of the columns in the dataset:
- `response_id`
- `submission_date`
- `satisfaction_score`
- `issue_description`
- `encountered_issue`
- `issue description`

Which of the requirements could each column be used to address?

In [ ]:
df = pd.read_csv(SMALL_DATA_PATH, parse_dates=['submission_date'])
print(f'Rows: {len(df)}  |  Columns: {list(df.columns)}')
df.head()

####Point of Concern #1: User Satisfaction Over Time

In the survey data, user satisfaction over time is measured on a scale from 1-5. We can calculate the average score to gauge users' general impressions about the tool.  

If we continuously plot the daily mean for this metric,  we can also get a sense of how/whether user experience varies naturally as well as how it varies before or after updates/changes to the chatbot.

In [ ]:
# Calculate the mean user satisfaction score by day
daily_sat = df.groupby('submission_date')['satisfaction_score'].mean().reset_index()
overall_avg = df['satisfaction_score'].mean()

fig = go.Figure()
# daily mean line
fig.add_trace(go.Scatter(
    x=daily_sat['submission_date'],
    y=daily_sat['satisfaction_score'],
    mode='lines+markers',
    line=dict(color='#028090', width=2.5),
    marker=dict(size=8),
    name='Daily avg'
))

# reference line for overall mean
fig.add_hline(
    y=overall_avg,
    line=dict(color='#D97706', width=1.5, dash='dash'),
    annotation_text=f'Overall avg: {overall_avg:.2f}',
    annotation_position='bottom right'
)

fig.update_layout(
    title='Chatbot Feedback (Small Dataset) - User Satisfaction Over Time',
    xaxis_title='Date',
    yaxis_title='Avg. Satisfaction Score (1-5)',
    yaxis=dict(range=[0, 5.5]),
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#E5E7EB')
fig.show()

print(f'Overall mean satisfaction: {overall_avg:.2f} / 5.0')

#### Point of Concern #2: Issue Rate Over Time
The small dataset we are working with has a field that tracks whether or not the user encountered an issue while using the chatbot.

The number of issues over time is a metric that was explicitly identified as important when constructing the user experience questionnaire. Because of the similarity of these metrics the `Issue Rate Over Time` can be visualized in a similar fashion as the `User Satisfaction Over Time`. Continuous monitoring and comparison of the two graphs may also help prioritize fixes to issues that matter the most to users.

In [ ]:
# calculate daily issue rate as a proportion of total responses that day
daily_issues = df.groupby('submission_date').agg(
    total=('encountered_issue', 'count'),
    issues=('encountered_issue', 'sum')
).reset_index()
daily_issues['issue_rate'] = daily_issues['issues'] / daily_issues['total']

overall_rate = df['encountered_issue'].mean()

fig = go.Figure()

# daily issue rate line
fig.add_trace(go.Scatter(
    x=daily_issues['submission_date'],
    y=daily_issues['issue_rate'],
    mode='lines+markers',
    line=dict(color='#BE123C', width=2.5),
    marker=dict(size=8),
    name='Daily rate'
))

# reference line shows overall rate across all days
fig.add_hline(
    y=overall_rate,
    line=dict(color='#D97706', width=1.5, dash='dash'),
    annotation_text=f'Overall rate: {overall_rate*100:.0f}%',
    annotation_position='bottom right'
)

fig.update_layout(
    title='Chatbot Feedback (Small Dataset) - Issues Per Day',
    xaxis_title='Date',
    yaxis_title='Issues per day',
    yaxis=dict(range=[0, 1]),
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#E5E7EB')
fig.show()

print(f'Total issues: {int(df["encountered_issue"].sum())} / {len(df)} responses ({overall_rate*100:.0f}%)')

####Point of Concern #3: Issue Descriptions
Because this dataset only contains 25 responses and only a fraction of those have descriptions of issues, employing statistical tools like word frequency counts or sentiment scoring to summarize responses is inadvisable.  In cases like this, it is best to conduct a manual review of the Issue Descriptions.  

Running the code cell below will display all unique issue descriptions in the dataset to make it easier to conduct this review.  In the next part of this demo, we will be working with much larger datasets where such statistical tools for insight are more appropriate.

In [ ]:
issues_df = df[df['encountered_issue'] == 1][['response_id', 'submission_date', 'issue_description']].copy()
issues_df = issues_df.dropna(subset=['issue_description'])

print(f'{len(issues_df)} issue descriptions\n')
for _, row in issues_df.iterrows():
    print(f"[{row['response_id']} | {row['submission_date'].date()}]")
    wrapped = textwrap.fill(row['issue_description'], width=80, initial_indent='  ', subsequent_indent='  ')
    print(wrapped)
    print()

---
## Part 2: Explaining The Context Window Problem
A LLM's context window is the maximum amount of text that the model can hold in "working memory" at any point in time.  All of the text contained within your prompt, any uploaded data, and the contents of your conversation history fill up the capacity of the model's context window.

When a context window is exceeded the model loses its capacity to consider all of the information (or context!) you have provided which can cause the model to:
- Silently analyze only a portion of the data you have entered
- Provide innacurate summaries and counts based on incomplete records without disclosing that the records are incomplete
- Generally inconsistent outputs
<br>

> **Tip:** Some AI tools will tell you if the context window has been exceeded, but many will not.

<br>

---

## Part 3: Large Dataset Exploration Using NLP for Reproducible Results
In this  part of the demo, you will have the opportunity to see what happens when context windows are exceeded using a large public comment dataset from NYC Central Business District Tolling Program Environmental Assessment.

**As a reminder, here are the questions we seek to answer:**
1. **(Overall sentiment toward the project)** What is the general sentiment toward the project across comments in the dataset
2. **(General concerns)**. What themes or concerns are associated with positive, negative, or neutral responses?
3. **(Fairness concerns)** Do commenters believe the proposed mitigation measures are sufficient for low-income drivers?

---
###Step 1: Review LLM Output for Large Dataset Summarization
- Prompt ChatGPT, Claude, Gemini or your chat-based LLM of choice with the following text.

- Be sure to attach the `chatbot_public_feedback.csv` file before hitting submit:
<br>

**Prompt Text:**
```
Based on this public comment data from the NYC Central Business District
Tolling Program Environmental Assessment, tell me about:

1) How people generally feel about the congestion pricing program overall
2) What concerns or themes are most commonly associated with positive,
   negative, or neutral responses
3) Whether commenters believe the proposed mitigation measures — such as
   the low-income discount plan, E-ZPass fee elimination, and overnight
   toll reduction — are sufficient to protect low-income drivers, and what
   additional relief (if any) they request
```
<br>

- After you get the output, follow up with a second prompt:

<br>

**Prompt Text:**
```
How did you arrive at the figures you created?
```  
<br>

- Note whether the tool can explain its methodology as well as what specific methodology was applied.  

- **Does that output seems appropriate in the context of your own experience analyzing survey results?**

### Step 2: Load in the Large Dataset
In this step, we'll simply be loading the large dataset into the current runtime environment.  Run the code cell below as-is.

Once the data has loaded, take note of the columns available to us:
- `submission_id` — groups individual comment records by submitter
- `comment_id` — unique identifier for each comment record
- `name` — name of the commenter
- `organization` — affiliated organization, if provided
- `date` — date the comment was submitted
- `comment_text` — the full text of the comment

The `comment_text` field is the primary input for all three analyses that follow.


In [ ]:
# load the public comment dataset
comments_df = pd.read_csv(LARGE_DATA_PATH, parse_dates=['date'])

# load the dataset; use format='mixed' to handle inconsistent date formats
# across rows (e.g. 'September 22, 2022' and '8/29/2022' both appear)
comments_df['date'] = pd.to_datetime(comments_df['date'], format='mixed')

print(f'Rows loaded: {len(comments_df)}')
print(f'Unique submissions: {comments_df["submission_id"].nunique()}')
print(f'Columns: {list(comments_df.columns)}')
print(f'Date range: {comments_df["date"].min().date()} to {comments_df["date"].max().date()}')
comments_df.head(3)

### Step 3: Summarize the Data

Now that the dataset is loaded, we will apply three NLP techniques to
address the questions outlined at the start of Part 3. Each technique
is introduced with a brief explanation in the markdown cell that precedes
its corresponding code.

The techniques we will employ are as follows:

1. **Sentiment Analysis** using [Valence Aware Dictionary and sEntiment
   Reasoner (VADER)](https://www.geeksforgeeks.org/python/python-sentiment-analysis-using-vader/)
   to assess overall sentiment distribution across the comment corpus.

2. **Topic Modeling** using [Latent Dirichlet Allocation (LDA)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)
   to surface recurring themes and identify which concerns are associated
   with positive, negative, and neutral responses.

3. **Word Frequency Analysis** applied to a filtered subset of comments
   referencing mitigation measures in order to identify which specific
   remedies commenters raise or request in the context of low-income driver
   fairness.

#### Point of Concern #1: Overall Sentiment Toward the Project

**What is the general sentiment toward the project across comments
in the dataset?**

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a
lexicon-based sentiment analysis tool well-suited to short, informal
text. It assigns each document a compound score ranging from -1
(most negative) to +1 (most positive). We apply the following
thresholds to classify each comment:

- **Positive**: compound score >= 0.05
- **Neutral**: compound score between -0.05 and 0.05
- **Negative**: compound score <= -0.05

> **Note about Processing Methodology:** Because individual submitters may have contributed multiple comment
records, we score at the `comment_id` level and then aggregate to
`submission_id` level using the mean compound score to avoid
over-representing prolific submitters.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

# score each comment record individually
comments_df['compound'] = comments_df['comment_text'].dropna().apply(
    lambda text: analyzer.polarity_scores(text)['compound']
)

# aggregate to submission level using mean compound score
submission_sentiment = (
    comments_df
    .groupby('submission_id')['compound']
    .mean()
    .reset_index()
    .rename(columns={'compound': 'mean_compound'})
)

# classify each submission using standard VADER thresholds
def classify(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

submission_sentiment['sentiment'] = submission_sentiment['mean_compound'].apply(classify)

# count submissions per sentiment class
counts = submission_sentiment['sentiment'].value_counts().reindex(
    ['Positive', 'Neutral', 'Negative']
)
total = counts.sum()

color_map = {
    'Positive': '#028090',
    'Neutral':  '#D97706',
    'Negative': '#BE123C',
}

fig = go.Figure(go.Bar(
    x=counts.index,
    y=counts.values,
    marker_color=[color_map[s] for s in counts.index],
    text=[f'{v} ({v/total*100:.1f}%)' for v in counts.values],
    textposition='outside',
))

fig.update_layout(
    title='CBD Tolling Program Public Comments - Sentiment Distribution by Submission',
    xaxis_title='Sentiment',
    yaxis_title='Number of Submissions',
    plot_bgcolor='white',
    yaxis=dict(range=[0, counts.max() * 1.2]),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#E5E7EB')
fig.show()

print('Sentiment breakdown (submission level):')
for label, count in counts.items():
    print(f'  {label}: {count} submissions ({count/total*100:.1f}%)')
print(f'\nMean compound score across all submissions: '
      f'{submission_sentiment["mean_compound"].mean():.3f}')

#### Point of Concern #2: General Concerns by Sentiment

**What themes or concerns are associated with positive, negative,
and neutral responses?**

Latent Dirichlet Allocation (LDA) is an unsupervised topic modeling
technique that identifies clusters of co-occurring words across a
corpus of documents. Each topic is represented as a probability
distribution over vocabulary terms, and each document is represented
as a mixture of topics.

In the steps below we:
1. Preprocess `comment_text` by removing stopwords and common
   project-specific filler terms that carry no analytic value
2. Fit a separate LDA model to the positive, neutral, and negative
   comment subsets identified in Question 1
3. Display the top terms per topic for each sentiment class to
   allow comparison of the thematic content associated with each

The number of topics (`n_topics`) is set to 4 per sentiment class.
You may adjust this parameter and re-run the cell to explore
alternative decompositions.

In [ ]:
N_TOPICS = 4
N_TOP_WORDS = 8

# stopwords: standard english list plus project-specific filler terms
EXTRA_STOPS = [
    'cbd', 'tolling', 'program', 'toll', 'mta', 'new', 'york',
    'manhattan', 'would', 'also', 'one', 'may', 'will', 'said',
    'program', 'project', 'street', 'city', 'area', 'like', 'make',
]

def get_top_words(model, feature_names, n=N_TOP_WORDS):
    """Return top n words for each topic in a fitted LDA model."""
    topics = {}
    for idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n - 1:-1]
        topics[f'Topic {idx + 1}'] = [feature_names[i] for i in top_indices]
    return topics

# merge sentiment labels back onto the comment-level dataframe
comments_with_sentiment = comments_df.merge(
    submission_sentiment[['submission_id', 'sentiment']],
    on='submission_id',
    how='left'
)

sentiment_classes = ['Positive', 'Neutral', 'Negative']
all_topics = {}

for sentiment in sentiment_classes:
    subset = comments_with_sentiment[
        comments_with_sentiment['sentiment'] == sentiment
    ]['comment_text'].dropna()

    # vectorize the text corpus for this sentiment class
    vectorizer = CountVectorizer(
        stop_words='english',
        max_df=0.90,
        min_df=3,
        max_features=1000,
    )

    # remove project-specific filler after standard stopword removal
    filtered = subset.str.lower().replace(
        '|'.join(EXTRA_STOPS), '', regex=True
    )

    dtm = vectorizer.fit_transform(filtered)
    feature_names = vectorizer.get_feature_names_out()

    # fit LDA model
    lda = LatentDirichletAllocation(
        n_components=N_TOPICS,
        random_state=42,
        max_iter=20,
    )
    lda.fit(dtm)

    all_topics[sentiment] = get_top_words(lda, feature_names)

# display top words per topic per sentiment class
for sentiment, topics in all_topics.items():
    print(f'\n--- {sentiment} Comments ---')
    for topic_label, words in topics.items():
        print(f'  {topic_label}: {", ".join(words)}')

##### Going Further: Deriving Descriptive Topic Labels

The raw LDA output requires the analyst to interpret each topic's word list
manually. A natural next step is to automate that labeling by passing the
top words to a transformer-based model — either using **zero-shot
classification** (`facebook/bart-large-mnli` via Hugging Face) to score
words against a predefined list of concern categories, or by passing the
word list directly to a chat-based LLM like Claude or GPT-4 via API to
generate a short descriptive label. No model training is required for
either approach.

#### Point of Concern #3: Fairness Concerns and Mitigation Sufficiency

**Do proposed mitigation measures align with commenter's concerns about impacts from the proposed improvements?**

According to the executive summary, the EA analyzes 18 resource areas to determine both benefits and potential adverse impacts from the proposed project.  For areas found to experience adverse effects the EA also explores mitigation measures.  

Those mitigation measures for environmental justice-specific concerns include:
| # | Mitigation Measure |
|---|---|
| 1 | Low-income discount plan |
| 2 | E-ZPass deposit fee elimination for customers without credit cards |
| 3 | Tax credit for CBD residents earning less than $60k/yr |
| 4 | Toll reduction for late-night traffic |
| 5 | Single toll cap for taxis and for-hire vehicles |
| 6 | Replacement of 500 diesel trucks in EJ communities with cleaner vehicles |
| 7 | Expansion of the NYCDOT off-hours delivery program to reduce traffic |
| 8 | Formation of an Environmental Justice Community Group for ongoing stakeholder engagement |
| 9 | Upgrades to school air filtration systems in affected areas |
| 10 | Additional funding for asthma case management in schools and establishment of a new asthma center in the Bronx |
| 11 | Renovations to parks and green spaces in affected communities |
| 12 | Roadside vegetation plantings in affected communities |

The question posed is complex, and the size of the dataset makes it difficult to summarize the many views and attitudes of countless stakeholders toward the program and proposed mitigation measures.

However, it is still possible to both adapt the methods we applied earlier in this part of the demo and construct additional metrics  that provide insight into both of these factors. Doing this requires coding knowledge, but chat-based LLMs can also be used to reduce development time by supporting debugging and code generation.  

**Run the code block below to:**
- Score every public comment with VADER sentiment (-1 to +1)
- Fuzzy-match comments to mitigation measures via keyword lists
- Aggregate mean sentiment per measure and prints validation samples
- Render a horizontal bar chart and console summary of results



In [ ]:
# VADER sentiment scorer
analyzer = SentimentIntensityAnalyzer()

# Match threshold (0–100); lower = more permissive, higher = stricter
SIMILARITY_THRESHOLD = 85

# Measure → keywords used to detect relevant comments via fuzzy match
MITIGATION_MEASURES = {
    'Low-Income Discount Plan':       ['low income discount', 'discount plan', 'low income toll'],
    'E-ZPass Fee Elimination':        ['ezpass', 'ez pass', 'e-zpass'],
    'NYC Tax Credit':                 ['tax credit', 'toll tax credit'],
    'Overnight Toll Reduction':       ['overnight toll', 'overnight rate', 'overnight period'],
    'Taxi / FHV Once-Per-Day Cap':    ['taxi cap', 'fhv cap', 'for hire vehicle cap',
                                       'once per day', 'one per day'],
    'NYC Clean Trucks Program':       ['clean truck', 'clean trucks program'],
    'Off-Hours Delivery Program':     ['off hours delivery', 'overnight delivery',
                                       'off hour deliveries'],
    'Environmental Justice Community Group': ['environmental justice community group',
                                              'ej community group'],
    'Air Filtration in Schools':      ['air filtration', 'filtration units in schools'],
    'Asthma Case Management':         ['asthma program', 'asthma case management',
                                       'asthma center'],
    'Park and Greenspace Renovation': ['park renovation', 'greenspace renovation',
                                       'green space'],
    'Roadside Vegetation':            ['roadside vegetation', 'tree planting',
                                       'roadside trees'],
}

# Score comments if not already scored; compound ranges -1 to +1
if 'compound' not in comments_df.columns:
    comments_df['compound'] = comments_df['comment_text'].dropna().apply(
        lambda text: analyzer.polarity_scores(text)['compound']
    )

def fuzzy_match(text, terms, threshold=SIMILARITY_THRESHOLD):
    """True if any term partially fuzzy-matches text at or above threshold."""
    text_lower = str(text).lower()
    result = process.extractOne(
        query=text_lower,
        choices=terms,
        scorer=fuzz.partial_ratio,
        score_cutoff=threshold,
    )
    return result is not None

# For each measure, mask matching comments and compute mean compound score
rows = []
for measure, terms in MITIGATION_MEASURES.items():
    mask = comments_df['comment_text'].apply(lambda text: fuzzy_match(text, terms))
    matched = comments_df.loc[mask, 'compound']
    if len(matched) == 0:
        continue
    rows.append({
        'measure':       measure,
        'mean_compound': round(matched.mean(), 3),
        'n_comments':    len(matched),
    })

# Sort ascending so most-negative measures appear at chart top

def sentiment_color(compound):
    """Map a VADER compound score to a bar color."""
    if compound >= 0.05:
        return '#22C55E'   # green  – positive
    elif compound <= -0.05:
        return '#EF4444'   # red    – negative
    else:
        return '#9CA3AF'   # gray   – neutral

measure_df = pd.DataFrame(rows).sort_values('mean_compound', ascending=True)
measure_df['color'] = measure_df['mean_compound'].apply(sentiment_color)

# Print 2 sample matches per measure to sanity-check threshold
print('Sample matched comments per measure (n=2) for threshold validation:\n')
for measure, terms in MITIGATION_MEASURES.items():
    mask = comments_df['comment_text'].apply(lambda text: fuzzy_match(text, terms))
    matched_texts = comments_df.loc[mask, 'comment_text'].head(2)
    print(f'  {measure} (n={mask.sum()}):')
    for text in matched_texts:
        print(f'    - {str(text)[:120]}...')
    print()

# Horizontal bar chart; bar length/color = mean sentiment, hover shows n_comments
fig = go.Figure(go.Bar(
    x=measure_df['mean_compound'],
    y=measure_df['measure'],
    orientation='h',
    marker_color=measure_df['color'],
    text=measure_df['mean_compound'].apply(lambda x: f'{x:.2f}'),
    textposition='outside',
    customdata=measure_df['n_comments'].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Mean sentiment: %{x:.3f}<br>'
        'Comments referencing measure: %{customdata}'
        '<extra></extra>'
    ),
))

# Dotted zero line separates positive from negative sentiment
fig.add_vline(x=0, line=dict(color='#6B7280', width=1, dash='dot'))

fig.update_layout(
    title=(
        f'CBD Tolling Program - Mean Comment Sentiment Toward Each Mitigation Measure '
        f'(fuzzy threshold={SIMILARITY_THRESHOLD})'
    ),
    xaxis_title='Mean VADER Compound Score',
    yaxis_title='Mitigation Measure',
    plot_bgcolor='white',
    height=550,
    margin=dict(l=220),  # wide left margin for long measure names
    xaxis=dict(range=[-1, 1]),
)
fig.update_xaxes(showgrid=True, gridcolor='#E5E7EB')
fig.update_yaxes(showgrid=False)
fig.show()

# Console summary with VADER standard thresholds: ±0.05 neutral band
print('Sentiment toward each mitigation measure:')
for _, row in measure_df.sort_values('mean_compound').iterrows():
    direction = (
        'positive' if row['mean_compound'] >= 0.05
        else 'negative' if row['mean_compound'] <= -0.05
        else 'neutral'
    )
    print(f'  {row["measure"]}: {row["mean_compound"]:.3f} ({direction}, n={row["n_comments"]})')